# HW3: Which trucks go to the workshop?

**Name:** SINA BAGHERZADEH KHIAVI &nbsp;&nbsp; **Student ID:** N26120261

See `README.md` for the scenario, requirements and grading. This starter loads the data, sets up MLflow, defines the cost, and writes the submission files; the rest is yours. Put your final model and threshold in `hw3_submission.py`.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"   # keep OpenMP threads modest on shared machines

import json
import warnings

import mlflow
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, make_scorer
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import hw3_submission as sub

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 0
C_FP, C_FN = 10, 500        # cost of an unnecessary workshop check / of a missed APS failure
EXPERIMENT = "hw3-truck-checks"
_MLFLOW_DB = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{_MLFLOW_DB}")  # browse: uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_experiment(EXPERIMENT)
rng = np.random.default_rng(SEED)

2026/11/25 19:06:18 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/25 19:06:18 INFO mlflow.store.db.utils: Updating database tables


2026/11/25 19:06:18 INFO mlflow.tracking.fluent: Experiment with name 'hw3-truck-checks' does not exist. Creating a new experiment.


In [2]:
aps = fetch_openml(data_id=41138, as_frame=True)
X = aps.data.astype("float32")                     # exactly what the grader passes to your pipeline
y = (aps.target == "pos").astype(int).to_numpy()   # 1 = failure in the air pressure system
print(X.shape, f"failures: {y.sum()} ({y.mean():.2%}); missing cells: {X.isna().mean().mean():.1%}")


def cost_per_truck(y_true, flagged) -> float:
    """Average cost per truck of a flagging decision (the grader uses the same definition)."""
    y_true, flagged = np.asarray(y_true), np.asarray(flagged).astype(bool)
    fp = np.sum(flagged & (y_true == 0))
    fn = np.sum(~flagged & (y_true == 1))
    return (C_FP * fp + C_FN * fn) / len(y_true)

(76000, 170) failures: 1375 (1.81%); missing cells: 8.3%


## 1. Protocol and metric

**Business costs.** Checking a healthy truck costs 10; missing a failing truck costs 500.

**Baselines (trivial policies):**
- *Never check*: flag nobody → pay 500 for every failure.
- *Check everything*: flag everyone → pay 10 for every truck.

**Selection metric.** Failures are rare (~2%), so accuracy is misleading. We select models by **average precision** (ranking quality under imbalance). Final decisions use **expected cost** after tuning the threshold.

**Protocol.** Hold out a final test set once. All tuning (hyperparameters + threshold) stays inside the training data. Nested CV estimates honest outer-fold performance. Evaluation is leakage-free.

In [3]:
# Hold out a final test set once; everything else stays in train for tuning.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Trivial policies on the held-out test set (baselines we must beat).
cost_never_check = cost_per_truck(y_test, np.zeros_like(y_test))
cost_check_all = cost_per_truck(y_test, np.ones_like(y_test))

SELECTION_METRIC = "average_precision"  # ranking under heavy imbalance; cost used after thresholding

print(f"train={X_train.shape[0]:,}  test={X_test.shape[0]:,}")
print(f"failures train={y_train.mean():.2%}  test={y_test.mean():.2%}")
print(f"baseline never-check cost/truck = {cost_never_check:.4f}")
print(f"baseline check-everything cost/truck = {cost_check_all:.4f}")
print(f"selection metric = {SELECTION_METRIC}")


def best_threshold_for_cost(y_true, proba, n_grid=101):
    """Pick the probability cutoff that minimizes expected cost (train/val only)."""
    thresholds = np.linspace(0.0, 1.0, n_grid)
    costs = [cost_per_truck(y_true, proba >= t) for t in thresholds]
    i = int(np.argmin(costs))
    return float(thresholds[i]), float(costs[i])


def paired_ttest(scores_a, scores_b):
    """Ordinary paired t-test on fold scores."""
    d = np.asarray(scores_a, dtype=float) - np.asarray(scores_b, dtype=float)
    return stats.ttest_rel(scores_a, scores_b)


def corrected_resampled_ttest(scores_a, scores_b, n_train, n_test):
    """Nadeau & Bengio corrected resampled t-test for overlapping CV folds."""
    d = np.asarray(scores_a, dtype=float) - np.asarray(scores_b, dtype=float)
    k = len(d)
    mean_d = d.mean()
    var_d = d.var(ddof=1)
    if var_d <= 0:
        return np.nan, 1.0
    t_stat = mean_d / np.sqrt((1.0 / k + n_test / n_train) * var_d)
    p = float(2 * stats.t.sf(abs(t_stat), df=k - 1))
    return float(t_stat), p

train=60,800  test=15,200
failures train=1.81%  test=1.81%
baseline never-check cost/truck = 9.0461
baseline check-everything cost/truck = 9.8191
selection metric = average_precision


## 2. Candidates under nested cross-validation (one MLflow run per outer fold)

We compare **logistic regression** vs **hist gradient boosting**.

- Outer CV = honest fold scores (one MLflow run each).
- Inner CV = hyperparameter search only on the outer-train split.
- Subsample the training data for nested CV so the notebook stays under ~20 minutes.

In [4]:
def make_logistic_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=SEED,
        )),
    ])


def make_hgb_pipeline():
    # HistGradientBoosting handles NaNs natively; class_weight helps rare failures.
    return HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=SEED,
        early_stopping=True,
        validation_fraction=0.1,
    )


CANDIDATES = {
    "logistic_regression": {
        "builder": make_logistic_pipeline,
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0],
        },
    },
    "hist_gradient_boosting": {
        "builder": make_hgb_pipeline,
        "param_grid": {
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 6],
            "max_iter": [100, 200],
        },
    },
}

# Subsample for nested CV runtime; final fit later uses the full training set.
N_NESTED = 12_000
if len(y_train) > N_NESTED:
    idx_sub, _ = train_test_split(
        np.arange(len(y_train)),
        train_size=N_NESTED,
        stratify=y_train,
        random_state=SEED,
    )
    X_nested = X_train.iloc[idx_sub].reset_index(drop=True)
    y_nested = y_train[idx_sub]
else:
    X_nested, y_nested = X_train, y_train

OUTER_SPLITS = 5
INNER_SPLITS = 2
outer_cv = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
ap_scorer = make_scorer(average_precision_score, response_method="predict_proba")

fold_run_ids = {name: [] for name in CANDIDATES}
fold_ap = {name: [] for name in CANDIDATES}
fold_cost = {name: [] for name in CANDIDATES}
n_train_outer = n_test_outer = None

for family, spec in CANDIDATES.items():
    print(f"\n=== nested CV: {family} ===")
    for fold_i, (tr_idx, te_idx) in enumerate(outer_cv.split(X_nested, y_nested)):
        X_tr, X_te = X_nested.iloc[tr_idx], X_nested.iloc[te_idx]
        y_tr, y_te = y_nested[tr_idx], y_nested[te_idx]
        n_train_outer, n_test_outer = len(tr_idx), len(te_idx)

        search = GridSearchCV(
            estimator=spec["builder"](),
            param_grid=spec["param_grid"],
            scoring=ap_scorer,
            cv=inner_cv,
            n_jobs=1,
            refit=True,
        )
        search.fit(X_tr, y_tr)
        best = search.best_estimator_

        proba = best.predict_proba(X_te)[:, 1]
        ap = float(average_precision_score(y_te, proba))

        # Threshold tuned on outer-train predictions only (no test leakage).
        proba_tr = best.predict_proba(X_tr)[:, 1]
        thr, _ = best_threshold_for_cost(y_tr, proba_tr)
        cost = float(cost_per_truck(y_te, proba >= thr))

        with mlflow.start_run(run_name=f"{family}_outer{fold_i}") as run:
            mlflow.set_tags({
                "family": family,
                "stage": "nested_outer",
                "fold": str(fold_i),
            })
            mlflow.log_params({k: str(v) for k, v in search.best_params_.items()})
            mlflow.log_param("threshold_tuned", thr)
            mlflow.log_param("n_train", n_train_outer)
            mlflow.log_param("n_test", n_test_outer)
            mlflow.log_metrics({
                "average_precision": ap,
                "expected_cost": cost,
            })
            run_id = run.info.run_id

        fold_run_ids[family].append(run_id)
        fold_ap[family].append(ap)
        fold_cost[family].append(cost)
        print(f"  fold {fold_i}: AP={ap:.4f}  cost={cost:.4f}  thr={thr:.2f}  run={run_id[:8]}")

print("\nMean outer AP:", {k: float(np.mean(v)) for k, v in fold_ap.items()})
print("Mean outer cost:", {k: float(np.mean(v)) for k, v in fold_cost.items()})


=== nested CV: logistic_regression ===


  fold 0: AP=0.5917  cost=1.7542  thr=0.26  run=b2ddc5be


  fold 1: AP=0.5404  cost=2.8750  thr=0.57  run=4b0f339a


  fold 2: AP=0.5487  cost=3.0917  thr=0.45  run=095483ca


  fold 3: AP=0.7421  cost=0.4167  thr=0.52  run=2b143930


  fold 4: AP=0.7461  cost=0.9542  thr=0.61  run=ac36e89e

=== nested CV: hist_gradient_boosting ===


  fold 0: AP=0.6649  cost=3.2042  thr=0.60  run=81958206


  fold 1: AP=0.7214  cost=2.4750  thr=0.64  run=f345ba39


  fold 2: AP=0.7373  cost=1.7958  thr=0.57  run=68a093b8


  fold 3: AP=0.8605  cost=1.1542  thr=0.43  run=1d049b2b


  fold 4: AP=0.8139  cost=1.3875  thr=0.55  run=53c584a6

Mean outer AP: {'logistic_regression': 0.63380240666106, 'hist_gradient_boosting': 0.7595914470640082}
Mean outer cost: {'logistic_regression': 1.8183333333333334, 'hist_gradient_boosting': 2.003333333333333}


## 3. Statistical comparison from the MLflow runs

Pull the paired outer-fold scores from MLflow and compare the two families with:
1. a plain paired t-test
2. the corrected resampled t-test (accounts for overlapping CV folds)

We do this for **average precision** (higher is better) and for **expected cost** (lower is better).

In [5]:
MODEL_A = "logistic_regression"
MODEL_B = "hist_gradient_boosting"
ALPHA = 0.05


def decide_better(scores_a, scores_b, higher_is_better, p_corrected, alpha=ALPHA):
    if p_corrected >= alpha or np.isnan(p_corrected):
        return "no_evidence"
    mean_a, mean_b = np.mean(scores_a), np.mean(scores_b)
    if higher_is_better:
        return MODEL_A if mean_a > mean_b else MODEL_B
    return MODEL_A if mean_a < mean_b else MODEL_B


def compare_metric(metric_name, scores_a, scores_b, higher_is_better):
    # Confirm scores match what is stored in MLflow for these run IDs.
    client = mlflow.tracking.MlflowClient()
    recovered_a, recovered_b = [], []
    for rid in fold_run_ids[MODEL_A]:
        recovered_a.append(client.get_run(rid).data.metrics[metric_name])
    for rid in fold_run_ids[MODEL_B]:
        recovered_b.append(client.get_run(rid).data.metrics[metric_name])
    assert np.allclose(recovered_a, scores_a)
    assert np.allclose(recovered_b, scores_b)

    t_naive = paired_ttest(scores_a, scores_b)
    t_corr, p_corr = corrected_resampled_ttest(
        scores_a, scores_b, n_train_outer, n_test_outer
    )
    better = decide_better(scores_a, scores_b, higher_is_better, p_corr)

    print(f"\n--- {metric_name} (higher_is_better={higher_is_better}) ---")
    print(f"mean {MODEL_A}: {np.mean(scores_a):.4f}")
    print(f"mean {MODEL_B}: {np.mean(scores_b):.4f}")
    print(f"paired t-test:      t={t_naive.statistic:.3f}  p={t_naive.pvalue:.4g}")
    print(f"corrected t-test:  t={t_corr:.3f}  p={p_corr:.4g}")
    print(f"better @ alpha={ALPHA}: {better}")

    return {
        "metric": metric_name,
        "higher_is_better": higher_is_better,
        "p_naive": float(t_naive.pvalue),
        "p_corrected": float(p_corr),
        "better_model": better,
        "fold_run_ids_a": list(fold_run_ids[MODEL_A]),
        "fold_run_ids_b": list(fold_run_ids[MODEL_B]),
        "n_train": int(n_train_outer),
        "n_test": int(n_test_outer),
    }


comparison_ap = compare_metric(
    "average_precision", fold_ap[MODEL_A], fold_ap[MODEL_B], higher_is_better=True
)
comparison_cost = compare_metric(
    "expected_cost", fold_cost[MODEL_A], fold_cost[MODEL_B], higher_is_better=False
)

# Primary comparison for answers.json: the ranking/selection metric.
comparison = {
    "model_a": MODEL_A,
    "model_b": MODEL_B,
    **comparison_ap,
}
print("\nPrimary comparison recorded for answers.json:", comparison["better_model"])


--- average_precision (higher_is_better=True) ---
mean logistic_regression: 0.6338
mean hist_gradient_boosting: 0.7596
paired t-test:      t=-4.899  p=0.008049
corrected t-test:  t=-3.266  p=0.0309
better @ alpha=0.05: hist_gradient_boosting

--- expected_cost (higher_is_better=False) ---
mean logistic_regression: 1.8183
mean hist_gradient_boosting: 2.0033
paired t-test:      t=-0.390  p=0.7165
corrected t-test:  t=-0.260  p=0.8077
better @ alpha=0.05: no_evidence

Primary comparison recorded for answers.json: hist_gradient_boosting


## 4. Threshold and final test

**Obstacle:** the default rule `probability >= 0.5` is badly wrong when missing a failure costs 50× more than an extra check.

We:
1. refit the chosen family on the full training set
2. measure cost at the default 0.5 threshold
3. tune a better threshold on training data only (CV out-of-fold probabilities)
4. evaluate once on the held-out test set

In [6]:
# Choose the model family: prefer corrected cost evidence, else AP evidence, else HGB.
if comparison_cost["better_model"] != "no_evidence":
    CHOSEN_MODEL = comparison_cost["better_model"]
elif comparison_ap["better_model"] != "no_evidence":
    CHOSEN_MODEL = comparison_ap["better_model"]
else:
    CHOSEN_MODEL = MODEL_B  # default to the stronger family when tests are inconclusive

print("Chosen model family:", CHOSEN_MODEL)

# Refit best hyperparams via inner CV on the full training set.
final_search = GridSearchCV(
    estimator=CANDIDATES[CHOSEN_MODEL]["builder"](),
    param_grid=CANDIDATES[CHOSEN_MODEL]["param_grid"],
    scoring=ap_scorer,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    n_jobs=1,
    refit=True,
)
final_search.fit(X_train, y_train)
final_model = final_search.best_estimator_
print("Best params:", final_search.best_params_)

# Out-of-fold probabilities on TRAIN only → tune threshold without touching test.
oof_estimator = CANDIDATES[CHOSEN_MODEL]["builder"]()
oof_estimator.set_params(**final_search.best_params_)
oof_proba = cross_val_predict(
    oof_estimator,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    method="predict_proba",
    n_jobs=1,
)[:, 1]

THRESHOLD, train_cost_at_thr = best_threshold_for_cost(y_train, oof_proba)
print(f"Tuned threshold (train OOF) = {THRESHOLD:.3f}  train cost ≈ {train_cost_at_thr:.4f}")

# Held-out test evaluation
test_proba = final_model.predict_proba(X_test)[:, 1]
cost_at_default = float(cost_per_truck(y_test, test_proba >= 0.5))
test_cost = float(cost_per_truck(y_test, test_proba >= THRESHOLD))
expected_cost = test_cost  # our estimate for new trucks = held-out cost at chosen rule
savings = float(cost_never_check - expected_cost)

with mlflow.start_run(run_name=f"final_{CHOSEN_MODEL}") as final_run:
    mlflow.set_tags({"family": CHOSEN_MODEL, "stage": "final_test"})
    mlflow.log_params({k: str(v) for k, v in final_search.best_params_.items()})
    mlflow.log_param("threshold", THRESHOLD)
    mlflow.log_metrics({
        "cost_never_check": float(cost_never_check),
        "cost_check_all": float(cost_check_all),
        "cost_at_default_threshold": cost_at_default,
        "test_cost_per_truck": test_cost,
        "expected_cost_per_truck": expected_cost,
        "expected_savings_per_truck": savings,
        "test_average_precision": float(average_precision_score(y_test, test_proba)),
    })
    FINAL_RUN_ID = final_run.info.run_id

print(f"\ncost never-check          = {cost_never_check:.4f}")
print(f"cost at default thr 0.5   = {cost_at_default:.4f}   << obstacle")
print(f"cost at tuned thr {THRESHOLD:.3f} = {test_cost:.4f}")
print(f"savings vs never-check    = {savings:.4f} per truck")
print(f"final MLflow run          = {FINAL_RUN_ID}")

# Persist for the submission module / grader
sub.THRESHOLD = THRESHOLD

def _build_chosen_pipeline():
    est = CANDIDATES[CHOSEN_MODEL]["builder"]()
    return est.set_params(**final_search.best_params_)

# Write hw3_submission.py so the grader gets an unfitted pipeline + threshold.
if CHOSEN_MODEL == "hist_gradient_boosting":
    params = final_search.best_params_
    _submission_src = f'''"""HW3 submission module. The autograder imports exactly these names.

The grader loads the data exactly as the starter does (170 float sensor columns with missing values),
fits build_pipeline() on its own training split, flags trucks with predict_proba(...)[:, 1] >= THRESHOLD
on a held-out split you do not see, and computes the cost per truck (10 per check, 500 per missed failure).
"""
from sklearn.base import BaseEstimator
from sklearn.ensemble import HistGradientBoostingClassifier

THRESHOLD: float = {THRESHOLD!r}


def build_pipeline() -> BaseEstimator:
    """Return an UNFITTED classifier (preprocessing included) that implements predict_proba."""
    return HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state={SEED},
        early_stopping=True,
        validation_fraction=0.1,
        learning_rate={params.get("learning_rate", 0.1)!r},
        max_depth={params.get("max_depth", None)!r},
        max_iter={params.get("max_iter", 100)!r},
    )
'''
else:
    C_val = final_search.best_params_.get("clf__C", 1.0)
    _submission_src = f'''"""HW3 submission module. The autograder imports exactly these names.

The grader loads the data exactly as the starter does (170 float sensor columns with missing values),
fits build_pipeline() on its own training split, flags trucks with predict_proba(...)[:, 1] >= THRESHOLD
on a held-out split you do not see, and computes the cost per truck (10 per check, 500 per missed failure).
"""
from sklearn.base import BaseEstimator
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

THRESHOLD: float = {THRESHOLD!r}


def build_pipeline() -> BaseEstimator:
    """Return an UNFITTED classifier (preprocessing included) that implements predict_proba."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            random_state={SEED},
            C={C_val!r},
        )),
    ])
'''

with open("hw3_submission.py", "w") as f:
    f.write(_submission_src)

import importlib
importlib.reload(sub)
print("wrote hw3_submission.py  THRESHOLD=", sub.THRESHOLD)

Chosen model family: hist_gradient_boosting


Best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 200}


Tuned threshold (train OOF) = 0.380  train cost ≈ 0.7714

cost never-check          = 9.0461
cost at default thr 0.5   = 0.8066   << obstacle
cost at tuned thr 0.380 = 0.7237
savings vs never-check    = 8.3224 per truck
final MLflow run          = 08c5122f6e7a4c59ad47cc2ef4915d61
wrote hw3_submission.py  THRESHOLD= 0.38


## 5. Check the submission module

In [7]:
try:
    model = sub.build_pipeline()
    print("module OK:", type(model).__name__, "threshold", sub.THRESHOLD)
except NotImplementedError as e:
    print("not done yet:", e)

module OK: HistGradientBoostingClassifier threshold 0.38


## 6. Answers and export

A reported number is `{"value": float, "run_id": "<MLflow run id>", "metric": "<metric name in that run>"}`.
Family names: `"logistic_regression"`, `"hist_gradient_boosting"`, `"random_forest"`, `"lightgbm"`, `"xgboost"`, `"other"`.

In [8]:
def reported(value, run_id, metric):
    return {"value": float(value), "run_id": run_id, "metric": metric}

# Deploy if we beat both trivial policies on the held-out test set.
deploy = (expected_cost < cost_never_check) and (expected_cost < cost_check_all)
memo_decision = "deploy_model" if deploy else "keep_current_policy"

justification = (
    f"We compared logistic regression and hist gradient boosting with nested CV and "
    f"corrected t-tests. Selected {CHOSEN_MODEL} and a cost-tuned threshold of {THRESHOLD:.3f} "
    f"(default 0.5 is far too strict given miss cost 500 vs check cost 10). "
    f"Held-out cost is {test_cost:.3f} per truck vs never-check {cost_never_check:.3f} "
    f"and check-all {cost_check_all:.3f}. Expected savings vs never-check: {savings:.3f} per truck. "
    f"Main caveat: the best threshold can drift if failure rates or costs change."
)

answers = {
    "student_id": "N26120261",
    "selection_metric": SELECTION_METRIC,
    "cost_never_check": float(cost_never_check),
    "cost_at_default_threshold": reported(cost_at_default, FINAL_RUN_ID, "cost_at_default_threshold"),
    "threshold": float(sub.THRESHOLD),
    "test_cost_per_truck": reported(test_cost, FINAL_RUN_ID, "test_cost_per_truck"),
    "expected_cost_per_truck": float(expected_cost),
    "chosen_model": CHOSEN_MODEL,
    "comparison": {
        "model_a": comparison["model_a"],
        "model_b": comparison["model_b"],
        "metric": comparison["metric"],
        "higher_is_better": comparison["higher_is_better"],
        "fold_run_ids_a": comparison["fold_run_ids_a"],
        "fold_run_ids_b": comparison["fold_run_ids_b"],
        "n_train": comparison["n_train"],
        "n_test": comparison["n_test"],
        "p_naive": comparison["p_naive"],
        "p_corrected": comparison["p_corrected"],
        "better_model": comparison["better_model"],
    },
    "memo": {
        "decision": memo_decision,
        "expected_savings_per_truck": float(savings),
        "main_risk": "threshold_drift",
        "justification": justification,
    },
}
with open("answers.json", "w") as f:
    json.dump(answers, f, indent=2)
mlflow.search_runs(experiment_names=[EXPERIMENT]).to_csv("mlflow_runs.csv", index=False)
print("wrote answers.json and mlflow_runs.csv")
print("decision:", memo_decision, " savings/truck:", round(savings, 4))

wrote answers.json and mlflow_runs.csv
decision: deploy_model  savings/truck: 8.3224


## AI-use statement

I used Cursor (Composer) to help set up the project environment, draft the nested-CV / MLflow / threshold-tuning notebook code, and explain homework terminology. I checked the output by running the notebook top-to-bottom, verifying baseline costs, confirming MLflow run IDs match `answers.json` / `mlflow_runs.csv`, and comparing held-out cost against the never-check and check-everything policies.